In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent  # if the notebook's working directory is notebooks/
sys.path.insert(0, str(repo_root))

from src.chunking import json_to_df_new

In [3]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [ ]:
def prepare_data(path, tokenizer, relevant_categories):
    chunks = json_to_df_new(path, tokenizer)
    chunks['categories'] = chunks['categories'].apply(
        lambda labels: [label for label in labels if label in relevant_categories] or ["None"]
    )
    return chunks

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "nlpaueb/bert-base-uncased-contracts"
labels = relevant_categories  # Keep the same order everywhere

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ContractDataset(Dataset):
    def __init__(self, chunks):
        # Partial chunks don't provide reliable positive/negative labels.
        if "label_status" in chunks.columns:
            chunks = chunks[chunks["label_status"] != "partial"]

        texts = chunks["text"].fillna("").astype(str).tolist()
        self.encodings = tokenizer(texts, truncation=True, max_length=512)

        self.targets = []
        for categories in chunks["categories"]:
            categories = set(categories or [])
            categories.discard("None")
            self.targets.append(
                [float(label in categories) for label in labels]
            )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        item = {
            key: value[index]
            for key, value in self.encodings.items()
        }
        item["labels"] = self.targets[index]
        return item

def collate_batch(features):
    targets = torch.tensor(
        [feature.pop("labels") for feature in features],
        dtype=torch.float,
    )
    batch = DataCollatorWithPadding(tokenizer)(features)
    batch["labels"] = targets
    return batch

project_root = Path.cwd().parent
train_chunks = prepare_data(project_root / "data/train.json", tokenizer, relevant_categories)
val_chunks = prepare_data(project_root / "data/val.json", tokenizer, relevant_categories)

train_dataset = ContractDataset(train_chunks)
val_dataset = ContractDataset(val_chunks)

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    problem_type="multi_label_classification",
    label2id=label2id,
    id2label=id2label,
)

args = TrainingArguments(
    output_dir="./contract-category-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_batch,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model("./contract-category-model")
tokenizer.save_pretrained("./contract-category-model")

KeyboardInterrupt: 